In [96]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [97]:
# Trainable Paramater를 가진 Custom Layer

# Fully connected network
class MyLinear(nn.Module):
    
    def __init__(
        self,
        in_units, # input dimension
        units,    # output dimension
    ):
        
        super().__init__()
        
        self.weight = nn.Parameter(
            torch.randn(
                in_units,
                units,
            )
        )
        
        self.bias = nn.Parameter(
            torch.randn(units)
        )
        
def my_linear_forward(
    self,
    X,
):
    # [B, in_units] @ [in_units, units] -> [B, units]
    linear = (
        X @ self.weight
        + self.bias
    )

    # ReLU(XW + b)
    return F.relu(linear)
    
MyLinear.forward = my_linear_forward

In [98]:
# Custom Layer의 Parameter & Gradient 확인

# [B, 5] -> [B, 3]
linear = MyLinear(
    in_units=5, # input dimension
    units=3,    # output dimension
)

print("[linear.weight]:", linear.weight.shape)
print(linear.weight)

print("\n[linear.bias]:", linear.bias.shape)
print(linear.bias)

# [2, 5] -> [2, 3]
X = torch.rand(2, 5)
Y = linear(X)

print("\n[Y = linear(X)]:", Y.shape)
print(Y)

# 역전파로 gradient 계산
Y.sum().backward()

print("\n[linear.weight.grad.shape]", linear.weight.grad.shape)
print("[linear.bias.grad.shape]", linear.bias.grad.shape)

[linear.weight]: torch.Size([5, 3])
Parameter containing:
tensor([[ 0.0343, -0.6067, -0.8415],
        [-1.5030, -1.7906,  0.1247],
        [-0.8905, -0.7938, -1.2300],
        [ 0.2171, -0.3266,  0.1559],
        [ 0.3725, -0.9954,  0.5060]], requires_grad=True)

[linear.bias]: torch.Size([3])
Parameter containing:
tensor([-0.5264, -0.4213,  1.4000], requires_grad=True)

[Y = linear(X)]: torch.Size([2, 3])
tensor([[0.0000, 0.0000, 1.5278],
        [0.0000, 0.0000, 1.0747]], grad_fn=<ReluBackward0>)

[linear.weight.grad.shape] torch.Size([5, 3])
[linear.bias.grad.shape] torch.Size([3])


In [101]:
# Custom Layer로 Model 구성

net = nn.Sequential(
    MyLinear(64, 8), # [B, 64] -> [B, 8]
    MyLinear(8, 1),  # [B, 8]  -> [B, 1]
)

# [2, 64] -> [2, 8] -> [2, 1]
X = torch.rand(2, 64)
Y = net(X)

print(net)
print(Y)
print(Y.shape)

Sequential(
  (0): MyLinear()
  (1): MyLinear()
)
tensor([[13.5739],
        [ 2.1790]], grad_fn=<ReluBackward0>)
torch.Size([2, 1])
